# THz-TCM Dual-View Manifold — One-Click Colab Runner & Rebuttal DL Baselines

- **Repository**: https://github.com/l114514sh-cmyk/THz-TCM-DualView-Manifold
- **Validation Protocol**: Identical to the locked SAA submission — Unified Single-Spectrum Physical QC (`1.2 <= mean(n) <= 2.6`, `mean(alpha) <= 55.0 cm^-1`), Outer 5-Fold Pellet-Grouped `StratifiedGroupKFold`, Inner 3-Fold Pellet-Grouped `StratifiedGroupKFold` (zero same-pellet leakage).
- **Recommended Colab Usage**:
  - **Free / T4**: Run `MODE = "smoke"` to verify pipeline & code changes at 0 compute-unit cost.
  - **Colab Pro / L4 or A100**: Set `MODE = "full"`, `INPUT_MODE = "both"`, `USE_DRIVE = True`, and `AUTO_DISCONNECT = True` for unattended runs with automatic checkpoint resume and zero idle burn.

In [ ]:
#@title 1. Hardware Check & Optional Google Drive Persistence
import os, subprocess, torch
from pathlib import Path

USE_DRIVE = False  #@param {type:"boolean"}
DRIVE_DIR = "/content/drive/MyDrive/THz_TCM_Colab_Runs"  #@param {type:"string"}

print(f"PyTorch: {torch.__version__} | CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,utilization.gpu", "--format=csv"], check=False)

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_DIR).mkdir(parents=True, exist_ok=True)
    OUTPUT_DIR = DRIVE_DIR
    print(f"Checkpoints & outputs will persist to Google Drive: {OUTPUT_DIR}")
else:
    OUTPUT_DIR = "/content/THz-TCM-DualView-Manifold/THz_TCM_Research/05_实验结果/模型结果/rebuttal_dl"
    print(f"Outputs will save locally to: {OUTPUT_DIR}")

In [ ]:
#@title 2. Clone / Pull Repository & Verify Locked .npz Spectral Caches
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/l114514sh-cmyk/THz-TCM-DualView-Manifold.git"
REPO_DIR = Path("/content/THz-TCM-DualView-Manifold")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)

caches = list((REPO_DIR / "THz_TCM_Research" / "01_原始实验数据").rglob("*_fast_cache.npz"))
print(f"Found {len(caches)} locked spectral caches:")
for c in sorted(caches):
    print("  -", c.relative_to(REPO_DIR))
assert len(caches) == 3, "Expected 3 *_fast_cache.npz files!"

In [ ]:
#@title 3. (Optional) Reproduce Locked Chemometrics & Leakage Audit Suite
RUN_LOCKED_CHEMOMETRICS = False  #@param {type:"boolean"}
RUN_LEAKAGE_AUDIT = False  #@param {type:"boolean"}

import subprocess
SCRIPT_DIR = REPO_DIR / "THz_TCM_Research" / "04_Python数据处理"

if RUN_LOCKED_CHEMOMETRICS:
    subprocess.run(["python", str(SCRIPT_DIR / "run_unified_double_nested_all.py")], check=True)
if RUN_LEAKAGE_AUDIT:
    subprocess.run(["python", str(SCRIPT_DIR / "run_strict_leakage_audit.py")], check=True)
if not RUN_LOCKED_CHEMOMETRICS and not RUN_LEAKAGE_AUDIT:
    print("Skipped locked CPU chemometrics suite (toggle checkboxes above to run).")

In [ ]:
#@title 4. Run Rebuttal Deep Learning Baselines (Pellet-Grouped Double-Nested CV)
MODE = "smoke"  #@param ["smoke", "full"]
INPUT_MODE = "raw_dual"  #@param ["raw_dual", "dual_view", "both"]
SEEDS = "42"  #@param {type:"string"}
FORCE_RERUN = False  #@param {type:"boolean"}

import subprocess
dl_script = REPO_DIR / "THz_TCM_Research" / "04_Python数据处理" / "run_rebuttal_dl_baselines.py"
cmd = [
    "python", str(dl_script),
    "--mode", MODE,
    "--input-mode", INPUT_MODE,
    "--output-dir", OUTPUT_DIR,
    "--seeds", *SEEDS.split()
]
if FORCE_RERUN:
    cmd.append("--force")

print("Executing:", " ".join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
#@title 5. Inspect Summary Table
from pathlib import Path
from IPython.display import Markdown, display

md_path = Path(OUTPUT_DIR) / f"Rebuttal_DL_Summary_{MODE}.md"
if md_path.exists():
    display(Markdown(md_path.read_text(encoding="utf-8")))
else:
    print(f"Summary not found yet: {md_path}")

In [ ]:
#@title 6. Compute-Unit Guard (Auto-Disconnect Runtime After Unattended Runs)
AUTO_DISCONNECT = False  #@param {type:"boolean"}

if AUTO_DISCONNECT:
    print("Unassigning Colab runtime to protect compute units...")
    from google.colab import runtime
    runtime.unassign()
else:
    print("AUTO_DISCONNECT is False. Remember to disconnect runtime when finished!")